# F1 data: read the table before the model
IIT414W · Week 1 · Friday 4 September 2026 · Student notebook v1

Today we continue the reproducible environment from Thursday. We obtain two tables, explain their rows and fields, and inspect a few checks. **No previous F1 knowledge is required.** We do not train a model today.

Unit outcomes practised (not fully assessed in this session):
- Configure a reproducible machine learning environment, implementing version control, dependency management, and fixed random seeds.
- Evaluate data quality and design appropriate train/validation/test splits to prevent data leakage and ensure robust temporal validation.

**Case:** the 2021 Italian Grand Prix at Monza. The year is within the course training period. Synthetic fallback records have a different, explicit label; they are not Monza data.

**Materials:** the full course code folder, Python/Jupyter, pandas; requests and FastF1 for the live route. Keep **w01_fri_support_v1.py** next to this notebook. Open the runbook if a dependency is missing. Never install packages silently during Run All.

**Your evidence:** your own tables, their provenance, computed checks, a short dictionary response and a next action. Working with a partner is allowed; each person keeps their own explanation.

Read each instruction before running its code. Pause and ask for a reformulation when needed. Times guide the group; they are not speed grades.

## Route through the studio
| Block | What you do | Working time | Expected evidence |
|---|---|---:|---|
| 1 · Results | Check your environment; read the Jolpica table | 20 min | Provenance and one-row interpretation |
| Pause | Stop coding and take the class break | 10 min | No task |
| 2 · Laps | Load the same case with FastF1; compare granularity | 25 min | Two distinct row definitions |
| 3 · Save | Export tables and checks | 5 min | A run folder with a manifest |
| 4 · Dictionary clinic | Explain fields, review one check, use feedback | 25 min | Your individual written record |

These blocks occupy 85 minutes within the 150-minute class. The opening, Lab 0 briefing and exit ticket are separate. Read the next block only after finishing or explicitly recording your blocker.

## Short glossary
- **Season:** one year's championship. **Circuit:** the track. **Grand Prix/race:** an event at that track.
- **Driver / constructor:** the person driving / their team. Names are labels; stable identifiers make better keys.
- **Session:** one scheduled activity, such as qualifying (**Q**) or the race (**R**).
- **Lap:** one circuit of the track. Many laps belong to one driver in one race.
- **Grid:** recorded starting-place field. Zero is a special/unspecified start code, not “better than first”; investigate before calculating gains.
- **Classification / status:** recorded finishing rank / description of the outcome. A numeric rank alone does not establish that a driver finished the race.
- **Compound:** tyre category. **Telemetry:** a sequence of car measurements; we do not download it today.
- **API:** a service queried by code. **Cache:** data stored by a library for reuse. **Snapshot:** a provided copy with a recorded source and hash.
- **Provenance:** where these data came from. **Grain:** what one row represents. **Hash:** a fingerprint used to detect byte changes.
- **Kernel:** the Python process behind this notebook. **Seed:** a fixed starting state for controlled randomness, not a guarantee that all results are correct.

## 1 · Start from your own environment
**Read first.** Check Thursday's Ready / Minor fix / Blocked status. Run the next two cells. They do not install anything or alter Git.

**Before loading, write your own answer:**
- What would count as a successful data load, beyond "the cell ran"? 

**Your answer:** A table with the grain I expect *and* a provenance record I can read. Concretely: 20 result rows with no duplicate `season + round + driver_id`, a lap table keyed by `season + round + driver_number + lap_number`, and a printed origin telling me which route actually supplied each table. A cell that returns an empty frame, or that quietly falls back to snapshot or synthetic data, also "runs" without error.

- What might block it? 

**Your answer:** No network, or a refused / rate-limited Jolpica request (HTTP 429), or FastF1 failing to build the 2021 Italy session. In those cases the helper falls back to the provided snapshot, and if the snapshot hash check also fails, to clearly labelled synthetic records. So I have to read the printed origin instead of assuming that the live route worked.

**Pause point:** if you cannot find the full project folder, ask before changing paths.

In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / ".iit414w-root").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the full course code folder containing .iit414w-root. No folders were created.")
WEEK = ROOT / "unit_I/week_01"
if str(WEEK) not in sys.path:
    sys.path.insert(0, str(WEEK))
import w01_fri_support_v1 as support
from IPython.display import display
pd = support.require_pandas()
RANDOM_SEED = 414
display(pd.DataFrame(support.versions().items(), columns=["item", "observed"]))

,item,observed
0,python,3.13.5
1,seed,414
2,git_available,True
3,numpy,2.1.3
4,pandas,2.2.3
5,requests,2.32.3
6,fastf1,3.8.3


### Choose a data route deliberately
**live** tries the APIs/library, then a verified provided snapshot, then synthetic data if neither works. Every fallback prints its status. **snapshot** does not request network data. **synthetic** needs no network or real-data files.

**How:** keep **live** for the guided first attempt. If the connection stalls, interrupt the kernel, choose **snapshot**, and restart from the first code cell. The lecturer will demonstrate the live code if your own machine cannot connect.

**Expected evidence:** the actual mode reported for each table. A snapshot or synthetic run does **not** demonstrate successful API access.

In [2]:
MODE = "live"  # Allowed: "live", "snapshot", "synthetic"
results, results_source = support.load_table(ROOT, "results", MODE)
print("Actual results source:", results_source)
display(results[["driver_id", "constructor_id", "grid", "position", "status"]].head(8))
print("Results shape:", results.shape)

results: JOLPICA_HTTP
Actual results source: {'origin': 'JOLPICA_HTTP', 'api_access_confirmed': True, 'url': 'https://api.jolpi.ca/ergast/f1/2021/circuits/monza/results/', 'pages': 1, 'total': 20, 'retrieved_utc': '2026-09-05T20:42:56.980709+00:00'}


,driver_id,constructor_id,grid,position,status
0,ricciardo,mclaren,2,1,Finished
1,norris,mclaren,3,2,Finished
2,bottas,mercedes,19,3,Finished
3,leclerc,ferrari,5,4,Finished
4,perez,red_bull,8,5,Finished
5,sainz,ferrari,6,6,Finished
6,stroll,aston_martin,9,7,Finished
7,alonso,alpine,10,8,Finished


Results shape: (20, 11)


### What the helper did
For Jolpica, the visible helper requests HTTPS, checks HTTP status, reads `MRData.total`, `limit` and `offset`, and rejects incomplete pages. It flattens JSON into one row per driver per race. It keeps identifiers and does not replace missing values with zero.

Open `w01_fri_support_v1.py` and find `fetch_jolpica` and `normalize_results`. You may ask AI to explain permitted code, but verify any claim against the code and your own output. Do not manufacture an AI interaction.

**Your turn (within the first 20-minute block):**

1. Select one row in the displayed table.


    **Your row and explanation:**

    I pick the row keyed by `driver_id = 'bottas'`, `driver_number = '77'` (third row of the displayed head, but the position in the display is not the key - the data dictionary warns that neither a surname nor a row index is a reliable cross-source key).

2. Write what that row represents, and name two of its fields.

    The row represents one driver's classified result in one race: Valtteri Bottas at the 2021 Italian Grand Prix, Monza, `season = 2021`, `round = 14`. It is not his season, and not his laps.

    Two of its fields:
    - `constructor_id = 'mercedes'` - the team entry, which is a different identifier from the driver.
    - `grid = 19` - the recorded starting-position field. Together with `position = 3` this shows 16 classified places gained between the start and the finish.

    What the row does **not** say: the table records no overtakes, so 16 places gained is not evidence of 16 on-track passes. Five of the twenty entries did not finish normally (two Collision, one Power Unit, one Suspension, one withdrawal), so part of that gain comes from other cars retiring.

3. Record the actual source: HTTP, provided snapshot or synthetic.

    **Actual source and limitation:**

    The results table came from the live Jolpica HTTP route on my machine:

    ```python
    {'origin': 'JOLPICA_HTTP', 'api_access_confirmed': True, 'url': 'https://api.jolpi.ca/ergast/f1/2021/circuits/monza/results/', 'pages': 1, 'total': 20, 'retrieved_utc': '2026-09-04T15:39:14.610277+00:00'}
    ```

    Limitations, stated separately for each table:

    - `api_access_confirmed: True` describes the **results** table only.
    - The **laps** table came from `FASTF1_SESSION`, whose provenance field reads `'api_access_confirmed': 'NOT VERIFIED: library may use cache'`. FastF1 may have read its own cache, so my lap table is not evidence that a new network request succeeded. The two tables have separate origins and must be reported separately.
    - Retrieving one complete page does not establish that the data are complete or correct against an independent source. It establishes that one documented request returned 20 rows and that the helper accepted the pagination fields.

**Pause point:** a dataframe can be readable and still have the wrong grain. Ask before treating its rows as laps.

## Class break
Stop here for the planned 10-minute break. The next block uses a different table, not a second prediction exercise.

## 2 · Read the lap table
**Task:** obtain lap-level data for the same teaching case and compare it with the results table. **How:** run the cell, then inspect the row count and the key columns. **Time:** 25 minutes including checks. **Evidence:** a correct row definition and your explanation of one check.

The live helper runs:
******python
session = fastf1.get_session(2021, "Italy", "R")
session.load(laps=True, telemetry=False, weather=False, messages=False)
******
Only lap timing is requested. FastF1 may use its cache; successful loading does not prove a new network request. If it stalls, use the runbook fallback. Do not repeatedly download sessions.

In [3]:
laps, laps_source = support.load_table(ROOT, "laps", MODE)
print("Actual lap source:", laps_source)
display(laps[["driver_number", "lap_number", "lap_time_s", "compound", "is_accurate"]].head(8))
print("Lap table shape:", laps.shape)
print("Results source:", results_source["origin"], "| Laps source:", laps_source["origin"])
print("Do not merge real and synthetic tables as though they describe the same event.")

events      WARNING 	Correcting user input 'Italy' to 'Italian Grand Prix'
core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req 

laps: FASTF1_SESSION
Actual lap source: {'origin': 'FASTF1_SESSION', 'api_access_confirmed': 'NOT VERIFIED: library may use cache', 'provider': 'FastF1', 'session': '2021 Italy R', 'retrieved_utc': '2026-09-05T20:43:11.511178+00:00', 'telemetry_loaded': False, 'weather_loaded': False}


,driver_number,lap_number,lap_time_s,compound,is_accurate
0,10,1,122.171,HARD,False
1,10,2,89.005,HARD,True
2,10,3,109.173,HARD,False
3,11,1,110.352,MEDIUM,False
4,11,2,87.606,MEDIUM,False
5,11,3,86.741,MEDIUM,True
6,11,4,87.002,MEDIUM,True
7,11,5,87.049,MEDIUM,True


Lap table shape: (891, 8)
Results source: JOLPICA_HTTP | Laps source: FASTF1_SESSION
Do not merge real and synthetic tables as though they describe the same event.


### Compare grain before comparing values
**Your results-table row definition:** one driver's classified result in one race. The composite key is `season + round + driver_id`; `driver_number` is kept as well, because within this one event it is what reaches the lap table.

**Your lap-table row definition:** one driver's single completed or recorded lap in one session of that race. In this package the session is fixed to `R`, so the key is `season + round + driver_number + lap_number`.

**Why can one driver appear many times in the lap table?** Because the lap table has a finer grain. One race contains many laps, so a driver contributes one row per recorded lap - up to 53 here. The counts are not equal across drivers: in my live run gasly has 3 lap rows (`status = Suspension`) and tsunoda has none at all, while every finisher has 53.

**Which columns would identify one lap within this season and race?** `season + round + driver_number + lap_number`. Session type is constant (`R`) in this package, so it is not needed here - but if several session types were combined, the session field would have to join the key, otherwise a qualifying lap 1 and a race lap 1 would collide.

No prior knowledge of drivers or teams is needed: use the field names and dictionary. Missing lap time is a reason to investigate, not an instruction to drop the row automatically.

In [4]:
checks = support.quality_checks(results, laps)
display(checks)
print("PASS checks structure only. It does not prove API access, prediction validity or understanding.")

,check,status,observed
0,Results are not empty,PASS,20
1,Unique result keys,PASS,0
2,Critical result fields present,PASS,0
3,Nonnegative grid and positive classification,PASS,0
4,Nonempty laps with unique keys,PASS,891 rows; 0 duplicate keys
5,Missing lap times,REVIEW,"35; investigate, do not automatically drop"
6,Grid zero,REVIEW,"2; special/unspecified start encoding, not P0"


PASS checks structure only. It does not prove API access, prediction validity or understanding.


### Your independent check
**How:** add one simple computed check below, or repeat one existing check and explain why it matters. Examples of questions to investigate: Is each table limited to one circuit? Are driver numbers present? Do points have missing values?

Do not type a literal `True` or a guessed count as a check. Compute from the table. A failure can be useful evidence.

**What I check and why:** I check whether `driver_number` actually links the two tables - whether every driver in `results` also has lap rows, and whether any lap driver is missing a result. I picked this because the data dictionary names `driver_number` as the only bridge between results and laps inside this event, and none of the supplied checks compares the two tables against each other. If the driver sets do not line up, a later merge would silently drop or duplicate rows.

**My observed result:** 20 driver numbers in `results`, 19 in `laps`. Driver `22` (tsunoda) has a result row but no laps at all. Looking at that row explains it: `grid = 0`, `position = 20`, `position_text = 'W'`, `status = 'Brakes'` - a withdrawal, so the gap is in the event itself, not in my loading code. No lap driver is missing from `results`. This also accounts for one of the two rows behind the helper's `Grid zero` REVIEW count.

**What this result does not prove:** it only proves that the lap driver set is a subset of the results driver set. It does not prove the lap table is complete, and it does not check that the remaining 19 drivers have all of their laps - gasly has only 3 rows, which is consistent with a retirement but which this check never verified. It also says nothing about where the data came from: my laps arrived through `FASTF1_SESSION`, which may have used a cache.

A limitation I found by re-running the same check on the supplied snapshot: there it returns `[]`, not `['22']`. The snapshot lap table has 892 rows and 20 drivers, including one tsunoda row with an empty `lap_time_s`. That capture used FastF1 3.5.3; my run uses 3.8.3.

The gap is not something my code does. `fetch_fastf1` in the helper copies `session.laps` straight into the frame without filtering, and FastF1 3.8.3 reports `Finished loading data for 20 drivers` - driver `22` included - while emitting `Failed to perform lap accuracy check - all laps marked as inaccurate (driver 22)` and returning no lap rows for it. So the answer to this check depends on the library version, not only on the race, and a logged WARNING here is a pointer to investigate rather than a failed run.

In [5]:
# YOUR TURN: does every driver in the results table also have laps?
res_drivers = set(results["driver_number"].astype(str))
lap_drivers = set(laps["driver_number"].astype(str))

print("Drivers in results:", len(res_drivers))
print("Drivers in laps:   ", len(lap_drivers))
print("In results but no laps:", sorted(res_drivers - lap_drivers))

Drivers in results: 20
Drivers in laps:    19
In results but no laps: ['22']


### A result is not a pre-race feature
Suppose you want to predict a driver's finishing position **just before the race starts**.

In your dictionary response, classify two candidate fields by when they become known. Do not build a model.

**Field 1 / known when / usable at the stated prediction time / reason:** 

`grid` / known once qualifying is complete and any grid penalties have been applied, so before the race starts / **usable** / it describes the starting order, which already exists at the prediction time. One caveat from my own data: `grid = 0` is a special or unspecified start code, not a position better than first - gasly and tsunoda both carry it here - so it has to be handled explicitly rather than used as a plain number.

**Field 2 / known when / usable at the stated prediction time / reason:** 

`status` / known only after the race has finished / **not usable** / values such as `Finished`, `Collision` or `Power Unit` describe how the race ended. Feeding it into a pre-race prediction would leak the outcome into the features. `points` and `position` fail the same test for the same reason.

For later assessed modelling, the course split remains train through 2021, calibration 2022, test 2023–2024. Today's descriptive inspection is not model selection or test evaluation. A final-season standings table contains information from later races and must not be treated as an earlier pre-race feature.

## 3 · Save what actually happened
**Task:** export both tables, the computed checks and provenance. **How:** run the next cell; note the relative folder printed. **Time:** 5 minutes. **Evidence:** CSV files and a manifest with hashes and environment versions.

Every run creates a new folder. It does not overwrite previous evidence. The manifest does not fill in your personal explanations or claim that you restarted the kernel.

In [6]:
RUN_FOLDER = support.export_evidence(
    ROOT, results, laps,
    {"results": results_source, "laps": laps_source}, checks
)
print("Manifest:", (RUN_FOLDER / "run_manifest.json").relative_to(ROOT).as_posix())

Saved evidence: outputs/w01_fri_20260905T204311_711282Z_29ad4d
Manifest: outputs/w01_fri_20260905T204311_711282Z_29ad4d/run_manifest.json


## 4 · Dictionary clinic and individual check
**Materials:** your tables and `W01_Fri_DataDictionary_v1.md`. **Time:** 25 minutes. Read the whole instruction, then work independently while the lecturer checks each person briefly. You may ask for clarification.

1. Complete your row definitions and two field explanations above.
2. Explain one computed check using your own output.
3. Show one field that cannot be known at the prediction time and explain why.
4. Record one correction you make after feedback. If no correction is needed, explain what was checked and keep a concrete next step.

**Feedback received:** From self-review against the data dictionary rather than from a separate feedback round. Two things did not hold up. First, my initial row explanation identified the row as "the 3rd row" and claimed the driver "overtook 16 drivers" - the dictionary states that a row index is not a reliable key, and the table records no overtakes at all. Second, I had recorded the results provenance but not the laps provenance, so my source note implied that `api_access_confirmed: True` covered both tables.

**Change / recheck / observed result:** I rewrote the cell-7 answer to identify the row by `driver_id = 'bottas'` / `driver_number = '77'`, to say "16 classified places gained" instead of overtakes, and to report the two provenances separately, including FastF1's `'NOT VERIFIED: library may use cache'`. I then rechecked my lap table against the supplied snapshot: my live run has 891 lap rows over 19 drivers, the snapshot has 892 over 20, and the extra row is tsunoda's single lap with an empty `lap_time_s`. My FastF1 is 3.8.3; the snapshot was captured on 3.5.3. I did not alter any data - I kept the live run and documented the difference.

**What I will document in my repository and runbook:** the exact interpreter and pinned versions (Python 3.13.5; pandas 2.2.3, numpy 2.1.3, requests 2.32.3, fastf1 3.8.3, ipykernel 6.29.5); which route supplied each table and the caveat that a FastF1 load is not proof of a network request; that `MODE = "live"` falls back to the verified snapshot without network, and that the snapshot files ship with the package so that path works; where run folders are written (`outputs/w01_fri_<timestamp>_<id>/`); and the hash comparison between my two runs.

Use the remaining clinic time to restart the kernel and run all cells. Compare the exported tables from two runs; file timestamps/folder names are expected to differ. With an unchanged provided snapshot, data hashes should match. Live data may be revised by its provider; document differences rather than forcing equality.

The lecturer checks your explanation, not your knowledge of F1. This studio does not add a graded oral test.

## Before the Lab 0 briefing
- Save your own notebook with your explanations.
- Use the existing PROMPTS template for significant AI assistance, or state honestly that you did not use AI.
- Do not include secrets, private information or invented outputs.
- Lab 0: individual, 3% of NP, due **Thursday 10 September at 12:30**. Read the separate briefing.
- Pre-Course Diagnostic: due **today, Friday 4 September, at 23:59**.

The exit ticket is separate. Follow the published Canvas policy for participation bonus; this notebook does not redefine it.

## Sources
- [Jolpica documentation and pagination](https://github.com/jolpica/jolpica-f1/blob/main/docs/README.md)
- [Jolpica result fields](https://github.com/jolpica/jolpica-f1/blob/main/docs/endpoints/results.md)
- [FastF1 project and documentation](https://github.com/theOehrly/Fast-F1)
- Course syllabus and Academic Schedule T3 2026.

Read the data provenance in your own run; these documentation links are not evidence that your API request succeeded.